In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
from testgen.prompts import Sensors

labels = Sensors.split("\n")
for i, sensor in enumerate(labels):
    labels[i] = sensor[sensor.find("(")+1: sensor.find(")")].strip()
    
labels = np.array(labels)

labels

array(['Acc', 'WSA', 'WS', 'YR', 'ST'], dtype='<U3')

In [4]:
# read results file
base_path = Path().cwd()
results_path = base_path.parent / "results"

available_results = list(results_path.glob("single*.json"))

# drop file with lower accuracy if multiple are found
available_results_dict = {}

for i, f in enumerate(available_results):
    t_ = f.name.split("_acc-")
    if available_results_dict.get(t_[0]) is None:
        available_results_dict[t_[0]] = (i, float(t_[1].split("_")[0]))
    else:
        if available_results_dict[t_[0]][1] < float(t_[1].split("_")[0]):
            del available_results[available_results_dict[t_[0]][0]]
            available_results_dict[t_[0]] = (i, float(t_[1].split("_")[0]))
        
    
available_results

[PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/single_gpt-4o-mini_n-1_acc-0.926_10.29.2024-10:46:10.json'),
 PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/single_gpt-4o-mini_n-3_acc-0.842_10.28.2024-09:55:42.json'),
 PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/single_gpt-4o-mini_n-5_acc-0.879_10.28.2024-10:03:41.json'),
 PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/single_gpt-4o-mini_n-8_acc-0.83_10.28.2024-10:10:08.json')]

In [5]:
def calc_scores(df):
    df_scores = pd.DataFrame(
        columns=["sensor", "accuracy", "precision", "recall", "f1"]
    )

    y_true = df["true_label"]
    y_pred = df["pred_label"]
    unique_labels = np.unique(y_true)
    unique_labels.sort()

    accuracy_score_ = round(accuracy_score(y_true, y_pred), 2)
    precision_score_ = round(
        precision_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )
    recall_score_ = round(
        recall_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )
    f1_score_ = round(
        f1_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )

    df_scores.loc[df_scores.shape[0] + 1] = [
        "All",
        accuracy_score_,
        precision_score_,
        recall_score_,
        f1_score_,
    ]

    for label in unique_labels:
        t_ = df[df["true_label"] == label]

        y_true_ = t_["true_label"]
        y_pred_ = t_["pred_label"]

        accuracy_score_ = round(accuracy_score(y_true_, y_pred_), 2)
        precision_score_ = round(
            precision_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )
        recall_score_ = round(
            recall_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )
        f1_score_ = round(
            f1_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )

        df_scores.loc[df_scores.shape[0] + 1] = [
            label,
            accuracy_score_,
            precision_score_,
            recall_score_,
            f1_score_,
        ]

    return df_scores.set_index("sensor")

In [6]:
def analyze(filename):

    type_, model_name, number_examples, *_ = filename.stem.split("_")
    number_examples = int(number_examples.split("-")[-1])

    file_under_investigation = results_path / filename
    with file_under_investigation.open("r") as f:
        data = json.load(f)
        
    # split responses and general stats
    responses = pd.DataFrame(data["responses"])
    responses.set_index("idx", inplace=True)
    del data["responses"]

    # collect stats per experiment
    stats = pd.DataFrame({k: [v]for k,v in data.items()})
    stats.insert(0, "type", type_)
    stats.insert(1, "model_name", model_name)
    stats.insert(2, "number_examples", number_examples)

    # collect label names for predictions and ground truth
    responses["true_label"] = responses["true_vector"].map(lambda x: " & ".join(labels[np.array(x[1:-1].split(",")).astype(bool)]) )
    responses["pred_label"] = responses["pred_vector"].map(lambda x: " & ".join(labels[np.array(x[1:-1].split(",")).astype(bool)]) )
    
    # accuracy per sensor
    summarize_ = responses.groupby(["true_label"]).aggregate({
        "accuracy": "sum",
        "true_label": "count"
    })

    summarize_.columns = ["true", "total"]
    summarize_["false"] = summarize_["total"] - summarize_["true"]


    all_vals = summarize_.sum(axis=0).values.tolist()
    summarize_.loc["All"] = all_vals

    summarize_.insert(0, "type", type_)
    summarize_.insert(1, "model_name", model_name)
    summarize_.insert(2, "number_examples", number_examples)

    df_scores = calc_scores(responses)

    summarize_ = summarize_.merge(df_scores, left_index=True, right_index=True).reset_index(names="sensors")
    
    return stats, summarize_

In [134]:
def plot_summarize(summarize_):
    ax = summarize_["accuracy"].plot.bar(
        title="Accuracy per Sensor",
        xlabel="",
    )

    _ = ax.bar_label(ax.containers[0])

In [138]:
stats = pd.DataFrame()
summarize = pd.DataFrame()

for filename in available_results:
    stats_, summarize_ = analyze(filename)
    
    stats = pd.concat([stats, stats_])
    summarize = pd.concat([summarize, summarize_])
    

stats.drop(columns="examples", inplace=True)
stats.set_index("number_examples", inplace=True)

summarize.reset_index(drop=True, inplace=True)

In [139]:
stats

,type,model_name,accuracy,number_of_reqs,total_tokens,total_completion_tokens,avg_token_per_req,avg_completion_token_per_req,avg_time_per_req
number_examples,,,,,,,,,
1,single,gpt-4o-mini,0.925926,189,149172,2457,789.269841,13.0,0.838779
3,single,gpt-4o-mini,0.841808,177,237421,2301,1341.361582,13.0,1.123236
5,single,gpt-4o-mini,0.878788,165,322098,2145,1952.109091,13.0,1.563024
8,single,gpt-4o-mini,0.829932,147,410300,1911,2791.156463,13.0,2.124535


In [140]:
summarize

,sensors,type,model_name,number_examples,true,total,false,accuracy,precision,recall,f1
0,Acc,single,gpt-4o-mini,1,29,31,2,0.94,1.00,0.94,0.97
1,Acc & WSA,single,gpt-4o-mini,1,25,25,0,1.00,1.00,1.00,1.00
2,ST,single,gpt-4o-mini,1,39,46,7,0.85,1.00,0.85,0.92
3,WS,single,gpt-4o-mini,1,26,26,0,1.00,1.00,1.00,1.00
4,WSA,single,gpt-4o-mini,1,28,33,5,0.85,1.00,0.85,0.92
5,YR,single,gpt-4o-mini,1,28,28,0,1.00,1.00,1.00,1.00
6,All,single,gpt-4o-mini,1,175,189,14,0.93,0.95,0.93,0.94
7,Acc,single,gpt-4o-mini,3,26,29,3,0.90,1.00,0.90,0.95
8,Acc & WSA,single,gpt-4o-mini,3,23,23,0,1.00,1.00,1.00,1.00
9,ST,single,gpt-4o-mini,3,31,44,13,0.70,1.00,0.70,0.83
